# LAB-D1-01: One-Neuron Boundary Workshop

**Purpose:** Make a neuron's weighted sum, sigmoid probability, threshold, and 2D decision-boundary geometry directly observable.

**Canonical objectives:** `OBJ-D1-01`, `OBJ-D1-02`, `OBJ-D1-03`  
**Estimated duration:** 40 minutes live; under 10 seconds compute  
**Prerequisites:** `LESSON-D1-01`, `LESSON-D1-02`, `ACT-D1-01`; basic NumPy indexing and 2D plots  
**Environment:** CPU only; local Python or Google Colab; NumPy and matplotlib; no network or download

You will follow **Observe -> Predict -> Modify -> Run -> Visualize -> Diagnose -> Explain -> Extend**. Restart the kernel and run from the top. The notebook is standalone and does not use state from another lab.

In [ ]:
import platform

import matplotlib
import matplotlib.pyplot as plt
import numpy as np

SEED = 11
rng = np.random.default_rng(SEED)
plt.rcParams.update({"figure.figsize": (7, 5), "axes.grid": True, "grid.alpha": 0.22})

print(f"Python {platform.python_version()} | NumPy {np.__version__} | matplotlib {matplotlib.__version__}")
print("Runtime target: CPU; no network access required.")

## Recap and Starter State

A neuron first computes the weighted score $z = w \cdot x + b$. A sigmoid can transform that score into a number between 0 and 1, and a threshold converts the number into a class prediction.

The boundary for threshold 0.5 is the set of points where $z=0$. The weight vector controls orientation and the bias changes location. Distance and parameter scale also affect probability confidence.

The starter data contains 12 fixed labeled points, a seeded 40-point linearly separable extension, and six unlabeled probes. The data and plot scaffolding are supplied; your implementation is limited to the neuron's core operations and manual parameter choice.

In [ ]:
X_fixed = np.array([
    [-2.0, -1.0], [-1.7, 0.2], [-1.2, -0.2], [-0.8, 0.1],
    [-0.5, 0.6], [-0.2, -0.8], [0.2, 0.1], [0.5, -0.4],
    [0.8, 0.0], [1.1, -0.8], [1.5, 0.2], [2.0, -0.6],
], dtype=float)
y_fixed = np.array([0, 0, 0, 0, 1, 0, 1, 0, 1, 0, 1, 1], dtype=int)

extension_0 = rng.normal(loc=(-1.1, -0.8), scale=(0.32, 0.30), size=(20, 2))
extension_1 = rng.normal(loc=(1.1, 0.8), scale=(0.32, 0.30), size=(20, 2))
X_extension = np.vstack([extension_0, extension_1])
y_extension = np.repeat([0, 1], 20)

probe_names = np.array(["west", "northwest", "center", "southeast", "east", "far-east"])
X_probes = np.array([[-1.6, -0.4], [-0.6, 0.8], [0.0, 0.0], [0.8, -0.5], [1.2, 0.4], [2.2, -0.2]])

assert X_fixed.shape == (12, 2)
assert X_extension.shape == (40, 2)
assert X_probes.shape == (6, 2)
print("Dataset ready: 12 fixed points, 40 extension points, and 6 unlabeled probes.")

In [ ]:
def plot_labeled_points(X, y, title, ax=None):
    if ax is None:
        _, ax = plt.subplots()
    for label, marker, name in [(0, "o", "class 0"), (1, "^", "class 1")]:
        mask = y == label
        ax.scatter(X[mask, 0], X[mask, 1], marker=marker, s=70, edgecolor="black", linewidth=0.7, label=name)
    for index, point in enumerate(X):
        ax.annotate(str(index), point, xytext=(5, 4), textcoords="offset points", fontsize=8)
    ax.set(xlabel="feature x1", ylabel="feature x2", title=title)
    ax.legend()
    return ax

plot_labeled_points(X_fixed, y_fixed, "Observe: the 12 fixed labeled points")
plt.show()

## Predict: Commit Before the Reveal

Without running a neuron yet, write a prediction for each isolated change:

1. Increase only `w1`. What should happen to boundary orientation?
2. Increase only `w2`. What should happen to boundary orientation?
3. Increase only `b`. Should the boundary rotate, translate, or both?
4. Multiply `w1`, `w2`, and `b` by the same positive number. What should happen to the 0.5 contour and to probability confidence away from it?
5. Circle two numbered fixed points that you expect to be hardest for an initial straight boundary.

Do not infer that sharper sigmoid confidence means better probability calibration. This visual studies geometry, not calibration.

In [ ]:
prediction_log = {
    "increase_w1": "",
    "increase_w2": "",
    "increase_bias": "",
    "scale_all_parameters": "",
    "two_hard_point_indices": "",
}
assert set(prediction_log) == {"increase_w1", "increase_w2", "increase_bias", "scale_all_parameters", "two_hard_point_indices"}
assert all(value.strip() for value in prediction_log.values()), (
    "Prediction checkpoint: replace every empty string with your own prediction before continuing."
)
print("Prediction checkpoint complete. Keep this record for the comparison plots.")

## Modify: Implement One Neuron

Complete only the marked `TODO` bodies.

- `weighted_sum`: accept a batch shaped `(n_examples, 2)` and return `(n_examples,)`.
- `sigmoid`: preserve the input shape and avoid overflow for large magnitudes.
- `probability_to_class`: apply the supplied threshold and return integer classes.
- `boundary_coefficients`: validate two weights and return coefficients in a form the contour helper can use without dividing by a weight.

Each placeholder raises `NotImplementedError` only when you call it. The setup cells above remain runnable in the starter notebook.

In [ ]:
def weighted_sum(X, weights, bias):
    """Return one weighted score per example."""
    # TODO: compute the vectorized weighted sum.
    raise NotImplementedError("TODO: return X @ weights + bias with shape (n_examples,)")


def sigmoid(scores):
    """Return numerically stable sigmoid probabilities."""
    # TODO: transform every score while preserving shape.
    raise NotImplementedError("TODO: implement a numerically stable sigmoid")


def probability_to_class(probabilities, threshold=0.5):
    """Return integer class predictions using the threshold."""
    # TODO: compare probabilities with threshold.
    raise NotImplementedError("TODO: threshold probabilities and return integer classes")


def boundary_coefficients(weights, bias):
    """Return validated coefficients for w1*x1 + w2*x2 + b = 0."""
    # TODO: validate exactly two finite weights and return weights, bias.
    raise NotImplementedError("TODO: validate and return boundary coefficients without dividing by a weight")

## Run: Six Probe Examples

After completing the functions, run the next cell. It reveals scores, probabilities, and predicted classes for six probes. Check shape and range before interpreting individual values.

In [ ]:
initial_weights = np.array([0.4, 0.3])
initial_bias = -0.1

probe_scores = weighted_sum(X_probes, initial_weights, initial_bias)
probe_probabilities = sigmoid(probe_scores)
probe_predictions = probability_to_class(probe_probabilities)

assert probe_scores.shape == (6,)
assert probe_probabilities.shape == (6,)
assert np.all((0.0 <= probe_probabilities) & (probe_probabilities <= 1.0))
assert probe_predictions.shape == (6,)

print(f"{'probe':<12} {'z':>8} {'probability':>13} {'class':>7}")
for name, score, probability, prediction in zip(probe_names, probe_scores, probe_probabilities, probe_predictions):
    print(f"{name:<12} {score:>8.3f} {probability:>13.3f} {prediction:>7d}")

## Visualize: Probability Shading and the 0.5 Contour

The plot exposes one relationship: how a straight score boundary relates to sigmoid probabilities and class predictions. Observe whether the 0.5 contour passes where $w \cdot x + b=0$ and which points lie near it.

Do **not** infer that the shading is a calibrated real-world probability. These parameters were chosen for geometry, not learned from representative data.

In [ ]:
def evaluate_neuron(X, weights, bias):
    scores = weighted_sum(X, weights, bias)
    probabilities = sigmoid(scores)
    predictions = probability_to_class(probabilities)
    return scores, probabilities, predictions


def decision_mesh(X, padding=0.6, resolution=180):
    x1 = np.linspace(X[:, 0].min() - padding, X[:, 0].max() + padding, resolution)
    x2 = np.linspace(X[:, 1].min() - padding, X[:, 1].max() + padding, resolution)
    xx1, xx2 = np.meshgrid(x1, x2)
    return xx1, xx2, np.column_stack([xx1.ravel(), xx2.ravel()])


def plot_neuron(X, y, weights, bias, title, ax=None, reference=None, annotate=False):
    if ax is None:
        _, ax = plt.subplots()
    checked_weights, checked_bias = boundary_coefficients(weights, bias)
    xx1, xx2, grid = decision_mesh(X)
    grid_scores = weighted_sum(grid, checked_weights, checked_bias).reshape(xx1.shape)
    grid_probabilities = sigmoid(grid_scores)
    shading = ax.contourf(xx1, xx2, grid_probabilities, levels=np.linspace(0, 1, 9), cmap="cividis", alpha=0.42)
    ax.contour(xx1, xx2, grid_scores, levels=[0.0], colors=["black"], linewidths=2)
    if reference is not None:
        reference_weights, reference_bias = reference
        reference_scores = weighted_sum(grid, reference_weights, reference_bias).reshape(xx1.shape)
        ax.contour(xx1, xx2, reference_scores, levels=[0.0], colors=["black"], linestyles="--", linewidths=1.4)
    plot_labeled_points(X, y, title, ax=ax)
    if annotate:
        for index, point in enumerate(X):
            ax.annotate(str(index), point, xytext=(5, 4), textcoords="offset points", fontsize=8)
    ax.text(0.02, 0.02, "solid: current 0.5 contour\ndashed: reference", transform=ax.transAxes, fontsize=8, va="bottom")
    return shading


def accuracy_for(X, y, weights, bias):
    return np.mean(evaluate_neuron(X, weights, bias)[2] == y)

In [ ]:
initial_scores, initial_probabilities, initial_predictions = evaluate_neuron(X_fixed, initial_weights, initial_bias)
initial_accuracy = np.mean(initial_predictions == y_fixed)
initial_misses = np.flatnonzero(initial_predictions != y_fixed)

fig, ax = plt.subplots(figsize=(8, 5.5))
shading = plot_neuron(X_fixed, y_fixed, initial_weights, initial_bias, "Run: initial probability field and boundary", ax=ax, annotate=True)
fig.colorbar(shading, ax=ax, label="sigmoid output")
plt.show()

print(f"Initial fixed-point accuracy: {initial_accuracy:.3f} ({np.sum(initial_predictions == y_fixed)}/12)")
print("Indices currently misclassified:", initial_misses.tolist())
assert initial_scores.shape == (12,)
assert len(initial_misses) == 2, "The supplied baseline should expose two near-boundary misses."

## Modify: Isolate One Parameter at a Time

Choose three comparison settings. Each must change only the named quantity. Then choose one positive common scale factor. Keep the initial setting as the dashed reference in every plot.

Before entering values, state which probe you will track and which visible property should remain unchanged in each comparison.

In [ ]:
w1_test_weights = None  # TODO: length-2 array; change only w1.
w2_test_weights = None  # TODO: length-2 array; change only w2.
bias_test_value = None  # TODO: scalar; keep both weights fixed.
scale_factor = None  # TODO: positive scalar other than 1.
tracked_probe_index = None  # TODO: integer from 0 through 5.

assert all(value is not None for value in [w1_test_weights, w2_test_weights, bias_test_value, scale_factor, tracked_probe_index]), (
    "Experiment checkpoint: replace every None with your own one-variable settings."
)
w1_test_weights = np.asarray(w1_test_weights, dtype=float)
w2_test_weights = np.asarray(w2_test_weights, dtype=float)
assert w1_test_weights.shape == (2,) and w2_test_weights.shape == (2,)
assert w1_test_weights[1] == initial_weights[1] and w1_test_weights[0] != initial_weights[0]
assert w2_test_weights[0] == initial_weights[0] and w2_test_weights[1] != initial_weights[1]
assert float(bias_test_value) != initial_bias
assert float(scale_factor) > 0 and not np.isclose(float(scale_factor), 1.0)
assert 0 <= int(tracked_probe_index) < len(X_probes)

In [ ]:
experiment_settings = [
    ("only w1 changed", w1_test_weights, initial_bias),
    ("only w2 changed", w2_test_weights, initial_bias),
    ("only bias changed", initial_weights.copy(), float(bias_test_value)),
    ("all parameters scaled", initial_weights * float(scale_factor), initial_bias * float(scale_factor)),
]

fig, axes = plt.subplots(2, 2, figsize=(12, 9), sharex=True, sharey=True)
base_probe_probability = probe_probabilities[int(tracked_probe_index)]
comparison_rows = []
for ax, (label, weights, bias) in zip(axes.ravel(), experiment_settings):
    plot_neuron(X_fixed, y_fixed, weights, bias, label, ax=ax, reference=(initial_weights, initial_bias))
    probability = evaluate_neuron(X_probes[[int(tracked_probe_index)]], weights, bias)[1][0]
    comparison_rows.append((label, accuracy_for(X_fixed, y_fixed, weights, bias), probability))
plt.tight_layout()
plt.show()

print(f"Tracked probe: {probe_names[int(tracked_probe_index)]}; baseline probability={base_probe_probability:.3f}")
for label, accuracy, probability in comparison_rows:
    print(f"{label:<24} accuracy={accuracy:.3f} tracked_probability={probability:.3f}")

## Diagnose: Geometry, Confidence, and Invariants

For each panel, record: one geometric change, one numerical change, and one invariant. Reconcile any prediction that did not match the evidence. Accuracy alone is not enough: two settings can classify the same finite points while producing different boundaries or probabilities.

In [ ]:
isolation_observations = {
    "only_w1": "",
    "only_w2": "",
    "only_bias": "",
    "common_scaling": "",
}
assert all(value.strip() for value in isolation_observations.values()), (
    "Diagnosis checkpoint: record evidence for all four comparisons before continuing."
)

## Diagnose a Change-Everything Attempt

The next supplied comparison changes both weights and the bias at once. Predict whether the result will let you attribute a rotation, translation, or probability change to one parameter. Run it, then explain what additional controlled comparison would discriminate between competing explanations.

In [ ]:
ambiguous_weights = initial_weights + np.array([0.7, -0.55])
ambiguous_bias = initial_bias + 0.45
fig, ax = plt.subplots(figsize=(7.5, 5.5))
plot_neuron(X_fixed, y_fixed, ambiguous_weights, ambiguous_bias, "Diagnose: several parameters changed", ax=ax, reference=(initial_weights, initial_bias))
plt.show()
print("This run is evidence of a combined change; it does not isolate a single cause.")

## Challenge: Manual Parameter Search

Open [ACT-D1-02 - Be the Neural Network](../challenges/day-1-challenges.md#act-d1-02---be-the-neural-network). Add at least two controlled attempts to the history below. Change one quantity between adjacent attempts when possible, and write your reason before running each attempt.

The fixed-point checkpoint accepts multiple settings. The goal is at least 10 of 12 correct, not a hidden target vector. Keep extension-set accuracy separate so a tiny fixed-set result does not masquerade as broad evidence.

In [ ]:
attempt_history = [
    {"label": "supplied baseline", "weights": initial_weights.copy(), "bias": initial_bias, "reason": "baseline"},
]
# TODO: append at least two dictionaries with label, length-2 weights, bias, and your reason.

assert len(attempt_history) >= 3, "Manual-search checkpoint: add at least two attempts."
evaluated_attempts = []
for attempt in attempt_history:
    weights = np.asarray(attempt["weights"], dtype=float)
    bias = float(attempt["bias"])
    assert weights.shape == (2,) and attempt["reason"].strip()
    fixed_accuracy = accuracy_for(X_fixed, y_fixed, weights, bias)
    extension_accuracy = accuracy_for(X_extension, y_extension, weights, bias)
    evaluated_attempts.append({**attempt, "weights": weights, "bias": bias, "fixed_accuracy": fixed_accuracy, "extension_accuracy": extension_accuracy})

best_attempt = max(evaluated_attempts, key=lambda item: item["fixed_accuracy"])
best_weights = best_attempt["weights"].copy()
best_bias = best_attempt["bias"]
best_accuracy = best_attempt["fixed_accuracy"]
for attempt in evaluated_attempts:
    print(f"{attempt['label']:<20} fixed={attempt['fixed_accuracy']:.3f} extension={attempt['extension_accuracy']:.3f} reason={attempt['reason']}")
assert best_accuracy >= 10 / 12, "Keep searching: the checkpoint is at least 10 of 12 fixed points."

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 5), sharex=True, sharey=True)
plot_neuron(X_fixed, y_fixed, best_weights, best_bias, "Best manual setting: fixed points", ax=axes[0], reference=(initial_weights, initial_bias), annotate=True)
plot_neuron(X_extension, y_extension, best_weights, best_bias, "Same setting: seeded extension", ax=axes[1], reference=(initial_weights, initial_bias))
for index in initial_misses:
    axes[0].scatter(*X_fixed[index], s=180, facecolors="none", edgecolors="red", linewidths=2)
plt.tight_layout()
plt.show()
print(f"Selected attempt: {best_attempt['label']} | fixed={best_accuracy:.3f} | extension={best_attempt['extension_accuracy']:.3f}")

## Diagnose the Two Baseline Misses

The red rings mark the two points missed by the supplied baseline. For each, compare score sign, distance from the contour, and target. Explain whether your manual change moved the boundary, changed orientation, changed off-boundary confidence, or combined effects. Then compare fixed-set and extension-set evidence.

## Break/Fix Check: A Vertical Boundary

Predict what happens if the second weight is exactly zero. A slope-intercept implementation would divide by that weight. The supplied visual instead contours the score grid, so the same strategy must handle vertical and near-vertical boundaries.

In [ ]:
vertical_weights = np.array([1.0, 0.0])
vertical_bias = -0.25
near_vertical_weights = np.array([1.0, 1e-12])
fig, axes = plt.subplots(1, 2, figsize=(11, 4.5), sharex=True, sharey=True)
plot_neuron(X_fixed, y_fixed, vertical_weights, vertical_bias, "Vertical: w2 = 0", ax=axes[0])
plot_neuron(X_fixed, y_fixed, near_vertical_weights, vertical_bias, "Near vertical: w2 = 1e-12", ax=axes[1])
plt.tight_layout()
plt.show()
checked_vertical = boundary_coefficients(vertical_weights, vertical_bias)
assert np.all(np.isfinite(checked_vertical[0])) and np.isfinite(checked_vertical[1])

## Explain and Reflect

Complete the evidence record. Cite a value or visual in every response; do not answer from vocabulary alone.

1. Which isolated setting changed location without changing orientation?
2. What stayed invariant under common positive scaling, and what changed?
3. Why was the change-everything run ambiguous?
4. What did the two baseline misses reveal?
5. Why does manual search become impractical as parameter count grows?

In [ ]:
reflection = {
    "location_without_rotation": "",
    "scaling_invariant_and_change": "",
    "ambiguity": "",
    "stubborn_points": "",
    "manual_search_limit": "",
}
assert all(value.strip() for value in reflection.values()), "Reflection checkpoint: complete all five evidence statements."
assert np.all(np.isfinite(sigmoid(np.array([-1000.0, 0.0, 1000.0]))))
assert np.all((probe_probabilities >= 0.0) & (probe_probabilities <= 1.0))
assert best_accuracy >= 10 / 12
scaled_scores = weighted_sum(X_probes, initial_weights * float(scale_factor), initial_bias * float(scale_factor))
assert np.array_equal(np.sign(probe_scores), np.sign(scaled_scores))
print("LAB-D1-01 checkpoint passed: shapes, ranges, manual criterion, scaling sign, and vertical contour strategy.")

## Troubleshooting

| Symptom | Likely cause | Recovery |
|---|---|---|
| Prediction checkpoint stops | One response is still empty | Write your own prediction in every field, then rerun |
| Matrix multiplication error | Weights are not shaped `(2,)` or examples are not rows | Print `X.shape` and `weights.shape`; restore batch-first orientation |
| Sigmoid warns about overflow | Direct exponentiation used the unstable sign | Split nonnegative and negative inputs or use an equivalent stable formula |
| Boundary code divides by zero | Slope-intercept form was used | Return affine coefficients and contour the score grid at zero |
| Comparison assertion stops | More than the named parameter changed | Reset to the supplied baseline and change one value only |
| Manual checkpoint stops | Too few attempts or fewer than 10 fixed points are correct | Log another reasoned attempt; do not erase prior evidence |

## Optional Extension

Change only the class threshold while holding the best weights and bias fixed. Predict which objects remain invariant, then compare class decisions and the probability field. Keep this separate from the core parameter experiment.

## Takeaways

- A one-neuron score has shape `(n_examples,)`; sigmoid preserves that shape.
- The `0.5` contour is the affine zero-score contour, including vertical cases.
- Isolated changes support causal attribution; simultaneous changes do not.
- Common positive scaling preserves the boundary while changing off-boundary probability values.
- Fixed-point success and extension-set evidence answer different questions.

Return to the [LAB-D1-01 debrief](../student-guide/day-1-student-guide.md#lab-d1-01---one-neuron-boundary-workshop). The prerequisite lesson and prediction activity are in [LESSON-D1-02](../student-guide/day-1-student-guide.md#lesson-d1-02---start-with-one-neuron) and [ACT-D1-01](../challenges/day-1-challenges.md#act-d1-01---boundary-motion-prediction). Environment and shape conventions are in the [shared environment](../../shared/environment.md) and [notation contract](../../shared/notation-and-style.md).